In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.dataset import ImageDataset
from torch.utils.data import DataLoader

annotations_file_trainval = Path("../data/preprocessed/trainval/annotations.csv")
img_dir_trainval = Path("../data/preprocessed/trainval/Images")

trainval_dataset = ImageDataset(annotations_file_trainval, img_dir_trainval)
trainval_dl = DataLoader(trainval_dataset, batch_size=32, shuffle=True)

In [3]:
from src.model import Model

model = Model()

In [4]:
from src.configs import S, B, C, IDX_TO_CLASS
from src.utils import convert_xywh_coords

def decode_preds(preds_batch):
    decoded_preds = []

    for pred in preds_batch:
        pred = pred.reshape((S, S, C + B * 5))
        objects = []

        for i in range(S):
            for j in range(S):
                pred_cell = pred[i][j]

                pred_class_idx = pred_cell[:20].argmax().item()
                pred_class_prob = pred_cell[pred_class_idx].item()
                pred_class = IDX_TO_CLASS[pred_class_idx]
                
                pred_1_confidence = (pred_cell[24].item() * \
                                     pred_class_prob,)
                pred_2_confidence = (pred_cell[29].item() * \
                                     pred_class_prob,)

                bbox_1 = convert_xywh_coords(pred_cell[20:24], i, j, False, True)
                bbox_2 = convert_xywh_coords(pred_cell[25:29], i, j, False, True)
                
                objects.append((pred_class,) + pred_1_confidence + bbox_1)
                objects.append((pred_class,) + pred_2_confidence + bbox_2)

        decoded_preds.append(objects)

    return decoded_preds

In [5]:
X_batch, y_batch = next(iter(trainval_dl))

X_batch.shape, y_batch.shape

(torch.Size([32, 3, 224, 224]), torch.Size([32, 7, 7, 30]))

In [6]:
preds = model(X_batch)
preds.shape

torch.Size([32, 1470])

In [7]:
preds = preds.reshape((preds.shape[0], S, S, B * 5 + C))
preds.shape

torch.Size([32, 7, 7, 30])

In [8]:
decoded_preds = decode_preds(preds)
len(decoded_preds), len(decoded_preds[0])

(32, 98)

In [9]:
CONFIDENCE_THRESHOLD = 0.375
from operator import itemgetter

def filter_sort(decoded_preds):
    sorted_preds = []

    # 1. filter and  by class
    for image in decoded_preds:
        valid_preds = {}
        for pred in image:
            if pred[1] > CONFIDENCE_THRESHOLD:
                class_name = pred[0]
                if class_name in valid_preds:
                    valid_preds[class_name].append(pred)
                else:
                    valid_preds[class_name] = [pred]
    
        sorted_preds.append(valid_preds)

    # 2. sort each class by confidence score 
    for image in sorted_preds:
        for class_name in image:
            image[class_name].sort(key=itemgetter(1), reverse=True)
    
    return sorted_preds

In [10]:
sorted_preds = filter_sort(decoded_preds)
sorted_preds

[{'cow': [('cow',
    0.46492605923561925,
    188.3707275390625,
    -86.9487075805664,
    69.98568725585938,
    90.81867218017578)],
  'bird': [('bird',
    0.41751353104168487,
    -78.1542739868164,
    73.5488052368164,
    48.94811248779297,
    -35.59485626220703),
   ('bird',
    0.4000424652627288,
    251.24317932128906,
    183.41030883789062,
    148.8401336669922,
    223.20333862304688)],
  'aeroplane': [('aeroplane',
    0.5592608228656957,
    58.000450134277344,
    192.34719848632812,
    -61.0908203125,
    182.19793701171875)]},
 {'motorbike': [('motorbike',
    0.4417887606767792,
    -27.864116668701172,
    185.5319061279297,
    41.93928146362305,
    89.98381042480469)],
  'horse': [('horse',
    0.4346560414758649,
    148.40509033203125,
    139.59222412109375,
    109.24188232421875,
    126.1975326538086)]},
 {'car': [('car',
    0.5048922745976014,
    254.41610717773438,
    187.34710693359375,
    129.44137573242188,
    163.56866455078125)]},
 {},
 {}

In [12]:
sorted_preds[0]['bird']

[('bird',
  0.41751353104168487,
  -78.1542739868164,
  73.5488052368164,
  48.94811248779297,
  -35.59485626220703),
 ('bird',
  0.4000424652627288,
  251.24317932128906,
  183.41030883789062,
  148.8401336669922,
  223.20333862304688)]

In [ ]:
final_preds = []
    
for image in sorted_preds:
    final_img_preds = {}

    for class_name, item in image.items():
        highest_conf = item.pop(0)
        final_img_preds[class_name] = [highest_conf]

    final_preds.append(final_img_preds)

In [ ]:
final_preds

In [13]:
from src.utils import IoU
from src.configs import NMS_IOU_THRESHOLD

def NMS(preds_batch):
    # 1. decode batch of predictions
    decoded_preds = decode_preds(preds_batch)

    # 2. filter, group, and sort the decoded predictions
    sorted_preds = filter_group_sort_preds(decoded_preds)

    # 3. perform Non-Maximum Suppression
    final_preds = []
    
    for image in sorted_preds:
        final_img_preds = {}

        for class_name, preds in image.items():
            final_img_preds[class_name] = []

            while preds: 
                highest_conf = preds.pop(0)
                final_img_preds.append(highest_conf)

                preds = [pred for pred in preds if IoU(highest_conf, pred) > NMS_IOU_THRESHOLD]

        final_preds.append(final_img_preds)

    return final_preds

In [14]:
final_preds = NMS(sorted_preds)
final_preds

AttributeError: 'dict' object has no attribute 'reshape'